# Feature engineering

### **Featue 1 - Revenue**

In [5]:
import pandas as pd

sales_df = pd.read_csv(
    "../data/processed/valid_sales.csv",
    dtype={"Invoice": "string"},
    parse_dates=["InvoiceDate"]
)

print(sales_df.head())
print(sales_df.dtypes)

  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  IsCancelled  \
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom        False   
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom        False   
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom        False   
3 2009-12-01 07:45:00   2.10      13085.0  United Kingdom        False   
4 2009-12-01 07:45:00   1.25      13085.0  United Kingdom        False   

   Revenue  Year  Month  Day  DayOfWeek  Hour  IsWeekend  
0     83.4  2009     12    1          1     7      False  
1     81.0  2009

### **Feature 2 - Time features**

In [6]:
sales_df["InvoiceDate"] = pd.to_datetime(
    sales_df["InvoiceDate"],
    errors="coerce"
)

sales_df["Year"] = sales_df["InvoiceDate"].dt.year
sales_df["Month"] = sales_df["InvoiceDate"].dt.month
sales_df["DayOfWeek"] = sales_df["InvoiceDate"].dt.dayofweek
sales_df["Hour"] = sales_df["InvoiceDate"].dt.hour

print(sales_df["InvoiceDate"].dtype)

print(sales_df["InvoiceDate"].isna().sum()) # output = 0 -> all InvoiceDate values were successfully converted.

datetime64[ns]
0


### **Feature 3 - Weekend indicator**

In [7]:
sales_df["IsWeekend"] = (
    sales_df["DayOfWeek"] >= 5
).astype(int)

### **Feature 4 - Customer RFM**

In [8]:
reference_date = sales_df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = (
    sales_df
    .groupby("Customer ID")
    .agg(
        Recency=(
            "InvoiceDate",
            lambda x: (reference_date - x.max()).days
        ),
        Frequency=("Invoice", "nunique"),
        Monetary=("Revenue", "sum")
    )
    .reset_index()
)

### **Feature 5 - Average order value**

In [9]:
customer_orders = (
    sales_df
    .groupby("Customer ID")
    .agg(
        TotalRevenue=("Revenue", "sum"),
        TotalOrders=("Invoice", "nunique")
    )
    .reset_index()
)

customer_orders["AverageOrderValue"] = (
    customer_orders["TotalRevenue"] /
    customer_orders["TotalOrders"]
)

### **Feature 6 - Unique products purchased**

In [10]:
unique_products = (
    sales_df
    .groupby("Customer ID")["StockCode"]
    .nunique()
    .reset_index(name="UniqueProducts")
)

## Forecasting features

#### For sales forecasting we need historical lag features.

### **Feature 7 - Previous day revenue**

In [11]:
daily_sales = pd.read_csv("../data/processed/daily_sales.csv")
forecast_df = daily_sales.copy()

forecast_df["Lag_1"] = (
    forecast_df["Revenue"].shift(1)
)

### **Feature 8 - Previous 7 day revenue**

In [12]:
forecast_df["Lag_7"] = (
    forecast_df["Revenue"].shift(7)
)

### **Feature 9 - Previous 14 day revenue**

In [13]:
forecast_df["Lag_14"] = (
    forecast_df["Revenue"].shift(14)
)

### **Feature 10 - Previous 28 day revenue**

In [14]:
forecast_df["Lag_28"] = (
    forecast_df["Revenue"].shift(28)
)

### **Rolling features**

In [15]:
forecast_df["RollingMean_7"] = (
    forecast_df["Revenue"]
    .shift(1)
    .rolling(7)
    .mean()
)

forecast_df["RollingMean_14"] = (
    forecast_df["Revenue"]
    .shift(1)
    .rolling(14)
    .mean()
)

forecast_df["RollingMean_28"] = (
    forecast_df["Revenue"]
    .shift(1)
    .rolling(28)
    .mean()
)